In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

# Dataset root
DATASET_ROOT = Path(
    "/content/drive/MyDrive/Visual_Product_Search/data/DeepFashion"
)

# Important paths
img_path = DATASET_ROOT / "Img"
eval_path = DATASET_ROOT / "Eval" / "list_eval_partition.txt"
anno_path = DATASET_ROOT / "Anno" / "list_bbox_inshop.txt"

print("Checking dataset structure...\n")

print("Img exists:", img_path.exists())
print("Eval file exists:", eval_path.exists())
print("Anno file exists:", anno_path.exists())

Checking dataset structure...

Img exists: True
Eval file exists: True
Anno file exists: True


In [5]:
!pip install -q \
transformers \
accelerate \
sentencepiece \
ultralytics \
hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.3 MB/s eta 0:00:00


In [6]:
import torch

print("PyTorch Version:")
print(torch.__version__)

print("\nCUDA Available:")
print(torch.cuda.is_available())

if torch.cuda.is_available():

    print("\nGPU Name:")
    print(torch.cuda.get_device_name(0))

    print("\nCUDA Device Count:")
    print(torch.cuda.device_count())

PyTorch Version:
2.10.0+cu128

CUDA Available:
True

GPU Name:
Tesla T4

CUDA Device Count:
1


In [7]:
from pathlib import Path

# ------------------------------------------------
# Project Root
# ------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Visual_Product_Search"
)

# ------------------------------------------------
# Dataset Paths
# ------------------------------------------------

DATASET_ROOT = (
    PROJECT_ROOT / "data" / "DeepFashion"
)

IMG_ROOT = (
    DATASET_ROOT / "Img"
)

EVAL_FILE = (
    DATASET_ROOT
    / "Eval"
    / "list_eval_partition.txt"
)

BBOX_FILE = (
    DATASET_ROOT
    / "Anno"
    / "list_bbox_inshop.txt"
)

# ------------------------------------------------
# Output Paths
# ------------------------------------------------

EMBEDDINGS_DIR = (
    PROJECT_ROOT / "embeddings"
)

INDEX_DIR = (
    PROJECT_ROOT / "indexes"
)

MODELS_DIR = (
    PROJECT_ROOT / "models"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT / "checkpoints"
)

OUTPUTS_DIR = (
    PROJECT_ROOT / "outputs"
)

# ------------------------------------------------
# Create output folders
# ------------------------------------------------

for folder in [
    EMBEDDINGS_DIR,
    INDEX_DIR,
    MODELS_DIR,
    CHECKPOINT_DIR,
    OUTPUTS_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------
# Verify Paths
# ------------------------------------------------

print("PROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nDATASET_ROOT exists:")
print(DATASET_ROOT.exists())

print("\nIMG_ROOT exists:")
print(IMG_ROOT.exists())

print("\nEVAL_FILE exists:")
print(EVAL_FILE.exists())

PROJECT_ROOT:
/content/drive/MyDrive/Visual_Product_Search

DATASET_ROOT exists:
True

IMG_ROOT exists:
True

EVAL_FILE exists:
True


In [8]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm import tqdm

# ------------------------------------------------
# Device
# ------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

# ------------------------------------------------
# Load CLIP
# ------------------------------------------------

model_name = "openai/clip-vit-base-patch32"

print("\nLoading CLIP model...")

model = CLIPModel.from_pretrained(
    model_name
).to(device)

processor = CLIPProcessor.from_pretrained(
    model_name
)

print("CLIP loaded successfully")

# ------------------------------------------------
# Load evaluation file
# ------------------------------------------------

with open(EVAL_FILE, "r") as f:
    lines = f.readlines()[2:]

data = []

for line in lines:

    parts = line.strip().split()

    data.append({
        "image_path": parts[0],
        "item_id": parts[1],
        "split": parts[2]
    })

df = pd.DataFrame(data)

# ------------------------------------------------
# Use FULL gallery
# ------------------------------------------------

gallery_df = df[df["split"] == "gallery"]

print("\nTotal Gallery Images:")
print(len(gallery_df))

# ------------------------------------------------
# Storage
# ------------------------------------------------

embeddings = []
image_paths = []
item_ids = []

# ------------------------------------------------
# Batch Processing
# ------------------------------------------------

batch_size = 32

for start_idx in tqdm(
    range(0, len(gallery_df), batch_size)
):

    batch_df = gallery_df.iloc[
        start_idx : start_idx + batch_size
    ]

    images = []

    valid_rows = []

    for _, row in batch_df.iterrows():

        try:

            img_path = (
                IMG_ROOT / row["image_path"]
            )

            image = Image.open(
                img_path
            ).convert("RGB")

            images.append(image)

            valid_rows.append(row)

        except Exception as e:

            print(f"Error: {img_path}")

    # Skip empty batches
    if len(images) == 0:
        continue

    # Preprocess
    inputs = processor(
        images=images,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    # Generate embeddings
    with torch.no_grad():

        batch_emb = model.get_image_features(
            **inputs
        )

    if hasattr(batch_emb, "pooler_output"):
        batch_emb = batch_emb.pooler_output

    batch_emb = F.normalize(
        batch_emb,
        p=2,
        dim=-1
    )

    batch_emb = (
        batch_emb.cpu().numpy()
    )

    # Store
    for emb, row in zip(
        batch_emb,
        valid_rows
    ):

        embeddings.append(emb)

        image_paths.append(
            row["image_path"]
        )

        item_ids.append(
            row["item_id"]
        )

# ------------------------------------------------
# Convert to numpy
# ------------------------------------------------

embeddings = np.array(embeddings)

print("\nEmbedding Matrix Shape:")
print(embeddings.shape)

# ------------------------------------------------
# Save embeddings
# ------------------------------------------------

embedding_file = (
    EMBEDDINGS_DIR
    / "gallery_embeddings_full.npy"
)

metadata_file = (
    EMBEDDINGS_DIR
    / "gallery_metadata_full.csv"
)

np.save(
    embedding_file,
    embeddings
)

metadata_df = pd.DataFrame({
    "image_path": image_paths,
    "item_id": item_ids
})

metadata_df.to_csv(
    metadata_file,
    index=False
)

print("\nEmbeddings saved successfully")

print("\nSaved files:")
print(embedding_file)
print(metadata_file)

Using device: cuda

Loading CLIP model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP loaded successfully

Total Gallery Images:
12612


100%|██████████| 395/395 [1:22:07<00:00, 12.48s/it]


Embedding Matrix Shape:
(12612, 512)

Embeddings saved successfully

Saved files:
/content/drive/MyDrive/Visual_Product_Search/embeddings/gallery_embeddings_full.npy
/content/drive/MyDrive/Visual_Product_Search/embeddings/gallery_metadata_full.csv


In [9]:
import hnswlib
import numpy as np
import pandas as pd

# ------------------------------------------------
# Load embeddings
# ------------------------------------------------

embedding_file = (
    EMBEDDINGS_DIR
    / "gallery_embeddings_full.npy"
)

metadata_file = (
    EMBEDDINGS_DIR
    / "gallery_metadata_full.csv"
)

embeddings = np.load(
    embedding_file
)

metadata = pd.read_csv(
    metadata_file
)

print("Embeddings shape:")
print(embeddings.shape)

# ------------------------------------------------
# Create HNSW index
# ------------------------------------------------

dim = embeddings.shape[1]

index = hnswlib.Index(
    space='cosine',
    dim=dim
)

# ------------------------------------------------
# Initialize index
# ------------------------------------------------

index.init_index(
    max_elements=len(embeddings),
    ef_construction=200,
    M=16
)

# ------------------------------------------------
# Add embeddings
# ------------------------------------------------

index.add_items(
    embeddings,
    np.arange(len(embeddings))
)

# ------------------------------------------------
# Search parameter
# ------------------------------------------------

index.set_ef(50)

print("\nHNSW index built successfully")

# ------------------------------------------------
# Test retrieval
# ------------------------------------------------

query_vector = embeddings[0]

labels, distances = index.knn_query(
    query_vector,
    k=5
)

print("\nTop Retrieval Results")
print("-" * 40)

for rank, (label, distance) in enumerate(
    zip(labels[0], distances[0]),
    start=1
):

    similarity = 1 - distance

    image_path = metadata.iloc[label][
        "image_path"
    ]

    print(
        f"{rank}. {image_path} "
        f"--> Similarity: {similarity:.4f}"
    )

# ------------------------------------------------
# Save index
# ------------------------------------------------

index_file = (
    INDEX_DIR
    / "fashion_hnsw_full.index"
)

index.save_index(
    str(index_file)
)

print("\nIndex saved successfully")

print("\nSaved index:")
print(index_file)

Embeddings shape:
(12612, 512)

HNSW index built successfully

Top Retrieval Results
----------------------------------------
1. img/WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg --> Similarity: 1.0000
2. img/WOMEN/Blouses_Shirts/id_00002264/02_1_front.jpg --> Similarity: 0.9395
3. img/WOMEN/Blouses_Shirts/id_00000740/03_4_full.jpg --> Similarity: 0.9387
4. img/WOMEN/Tees_Tanks/id_00007691/04_3_back.jpg --> Similarity: 0.9312
5. img/WOMEN/Blouses_Shirts/id_00000001/02_3_back.jpg --> Similarity: 0.9308

Index saved successfully

Saved index:
/content/drive/MyDrive/Visual_Product_Search/indexes/fashion_hnsw_full.index


In [10]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm import tqdm
import hnswlib

# ------------------------------------------------
# Device
# ------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

# ------------------------------------------------
# Load CLIP
# ------------------------------------------------

model_name = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(
    model_name
).to(device)

processor = CLIPProcessor.from_pretrained(
    model_name
)

print("CLIP loaded successfully")

# ------------------------------------------------
# Load metadata
# ------------------------------------------------

metadata_file = (
    EMBEDDINGS_DIR
    / "gallery_metadata_full.csv"
)

metadata = pd.read_csv(
    metadata_file
)

# ------------------------------------------------
# Load embeddings
# ------------------------------------------------

embedding_file = (
    EMBEDDINGS_DIR
    / "gallery_embeddings_full.npy"
)

embeddings = np.load(
    embedding_file
)

print("\nEmbeddings loaded:")
print(embeddings.shape)

# ------------------------------------------------
# Build HNSW index
# ------------------------------------------------

dim = embeddings.shape[1]

index = hnswlib.Index(
    space='cosine',
    dim=dim
)

index.load_index(
    str(
        INDEX_DIR
        / "fashion_hnsw_full.index"
    )
)

index.set_ef(50)

print("\nHNSW index loaded")

# ------------------------------------------------
# Read evaluation file
# ------------------------------------------------

with open(EVAL_FILE, "r") as f:
    lines = f.readlines()[2:]

data = []

for line in lines:

    parts = line.strip().split()

    data.append({
        "image_path": parts[0],
        "item_id": parts[1],
        "split": parts[2]
    })

df = pd.DataFrame(data)

# ------------------------------------------------
# Query subset
# ------------------------------------------------

query_df = (
    df[df["split"] == "query"]
    .head(1000)
)

print("\nQueries selected:")
print(len(query_df))

# ------------------------------------------------
# Recall@5
# ------------------------------------------------

k = 5
correct = 0

for _, row in tqdm(
    query_df.iterrows(),
    total=len(query_df)
):

    try:

        img_path = (
            IMG_ROOT / row["image_path"]
        )

        image = Image.open(
            img_path
        ).convert("RGB")

        # Preprocess
        inputs = processor(
            images=image,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        # Embedding
        with torch.no_grad():

            emb = model.get_image_features(
                **inputs
            )

        if hasattr(emb, "pooler_output"):
            emb = emb.pooler_output

        emb = F.normalize(
            emb,
            p=2,
            dim=-1
        )

        emb = emb.cpu().numpy()

        # Search
        labels, distances = index.knn_query(
            emb,
            k=k
        )

        retrieved_items = metadata.iloc[
            labels[0]
        ]["item_id"].values

        # Check correctness
        if row["item_id"] in retrieved_items:
            correct += 1

    except Exception as e:

        print(f"Error: {img_path}")

# ------------------------------------------------
# Final Recall
# ------------------------------------------------

recall_at_5 = correct / len(query_df)

print(f"\nRecall@5: {recall_at_5:.4f}")

Using device: cuda


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded successfully

Embeddings loaded:
(12612, 512)

HNSW index loaded

Queries selected:
1000


100%|██████████| 1000/1000 [06:34<00:00,  2.53it/s]


Recall@5: 0.6430


In [11]:
import pandas as pd

# ------------------------------------------------
# Read evaluation file
# ------------------------------------------------

with open(EVAL_FILE, "r") as f:
    lines = f.readlines()[2:]

data = []

for line in lines:

    parts = line.strip().split()

    data.append({
        "image_path": parts[0],
        "item_id": parts[1],
        "split": parts[2]
    })

df = pd.DataFrame(data)

# ------------------------------------------------
# Train split only
# ------------------------------------------------

train_df = df[
    df["split"] == "train"
]

print("Train images:")
print(len(train_df))

# ------------------------------------------------
# Group by item_id
# ------------------------------------------------

grouped = train_df.groupby(
    "item_id"
)

pairs = []

# ------------------------------------------------
# Create positive pairs
# ------------------------------------------------

for item_id, group in grouped:

    images = group["image_path"].tolist()

    # Need at least 2 images
    if len(images) < 2:
        continue

    # Create adjacent pairs
    for i in range(len(images) - 1):

        pairs.append({
            "img1": images[i],
            "img2": images[i + 1],
            "item_id": item_id
        })

# ------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------

pairs_df = pd.DataFrame(pairs)

print("\nPositive pairs created:")
print(len(pairs_df))

print("\nSample pairs:")
print(pairs_df.head())

Train images:
25882

Positive pairs created:
21885

Sample pairs:
                                           img1  \
0  img/WOMEN/Dresses/id_00000002/02_1_front.jpg   
1   img/WOMEN/Dresses/id_00000002/02_2_side.jpg   
2   img/WOMEN/Dresses/id_00000002/02_4_full.jpg   
3   img/WOMEN/Skirts/id_00000003/02_1_front.jpg   
4    img/WOMEN/Skirts/id_00000003/02_2_side.jpg   

                                                img2      item_id  
0        img/WOMEN/Dresses/id_00000002/02_2_side.jpg  id_00000002  
1        img/WOMEN/Dresses/id_00000002/02_4_full.jpg  id_00000002  
2  img/WOMEN/Dresses/id_00000002/02_7_additional.jpg  id_00000002  
3         img/WOMEN/Skirts/id_00000003/02_2_side.jpg  id_00000003  
4         img/WOMEN/Skirts/id_00000003/02_3_back.jpg  id_00000003  


In [14]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch

# ------------------------------------------------
# Small subset for first training
# ------------------------------------------------

train_pairs_df = pairs_df.head(5000)

print("Training pairs selected:")
print(len(train_pairs_df))

# ------------------------------------------------
# Retrieval Dataset
# ------------------------------------------------

class FashionRetrievalDataset(Dataset):

    def __init__(
        self,
        pairs_df,
        image_root,
        processor
    ):

        self.pairs_df = pairs_df
        self.image_root = image_root
        self.processor = processor

    def __len__(self):

        return len(self.pairs_df)

    def __getitem__(self, idx):

        row = self.pairs_df.iloc[idx]

        img1_path = (
            self.image_root / row["img1"]
        )

        img2_path = (
            self.image_root / row["img2"]
        )

        # Load images
        image1 = Image.open(
            img1_path
        ).convert("RGB")

        image2 = Image.open(
            img2_path
        ).convert("RGB")

        # Preprocess
        inputs1 = self.processor(
            images=image1,
            return_tensors="pt"
        )

        inputs2 = self.processor(
            images=image2,
            return_tensors="pt"
        )

        return {
            "pixel_values_1":
                inputs1["pixel_values"][0],

            "pixel_values_2":
                inputs2["pixel_values"][0]
        }

# ------------------------------------------------
# Create dataset
# ------------------------------------------------

dataset = FashionRetrievalDataset(
    train_pairs_df,
    IMG_ROOT,
    processor
)

# ------------------------------------------------
# Create dataloader
# ------------------------------------------------

dataloader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

print("\nDataset and DataLoader ready")

# ------------------------------------------------
# Test one batch
# ------------------------------------------------

batch = next(iter(dataloader))

print("\nBatch shapes:")

print(
    batch["pixel_values_1"].shape
)

print(
    batch["pixel_values_2"].shape
)

Training pairs selected:
5000

Dataset and DataLoader ready

Batch shapes:
torch.Size([16, 3, 224, 224])
torch.Size([16, 3, 224, 224])


In [15]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm

# ------------------------------------------------
# Device
# ------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------
# Load CLIP
# ------------------------------------------------

model_name = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(
    model_name
).to(device)

print("CLIP loaded")

# ------------------------------------------------
# Optimizer
# ------------------------------------------------

optimizer = AdamW(
    model.parameters(),
    lr=1e-5
)

# ------------------------------------------------
# Training mode
# ------------------------------------------------

model.train()

# ------------------------------------------------
# Training config
# ------------------------------------------------

epochs = 1

# ------------------------------------------------
# Training loop
# ------------------------------------------------

for epoch in range(epochs):

    total_loss = 0

    progress_bar = tqdm(dataloader)

    for batch in progress_bar:

        # Move to GPU
        pixel_values_1 = batch[
            "pixel_values_1"
        ].to(device)

        pixel_values_2 = batch[
            "pixel_values_2"
        ].to(device)

        # ------------------------------------------------
        # Generate embeddings
        # ------------------------------------------------

        emb1 = model.get_image_features(
            pixel_values=pixel_values_1
        )

        emb2 = model.get_image_features(
            pixel_values=pixel_values_2
        )

        if hasattr(emb1, "pooler_output"):
            emb1 = emb1.pooler_output

        if hasattr(emb2, "pooler_output"):
            emb2 = emb2.pooler_output

        # ------------------------------------------------
        # Normalize
        # ------------------------------------------------

        emb1 = F.normalize(
            emb1,
            p=2,
            dim=-1
        )

        emb2 = F.normalize(
            emb2,
            p=2,
            dim=-1
        )

        # ------------------------------------------------
        # Cosine similarity
        # ------------------------------------------------

        similarity = F.cosine_similarity(
            emb1,
            emb2
        )

        # ------------------------------------------------
        # Loss
        # ------------------------------------------------

        loss = 1 - similarity.mean()

        # ------------------------------------------------
        # Backpropagation
        # ------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        # ------------------------------------------------
        # Logging
        # ------------------------------------------------

        total_loss += loss.item()

        progress_bar.set_description(
            f"Loss: {loss.item():.4f}"
        )

    avg_loss = total_loss / len(dataloader)

    print(
        f"\nEpoch {epoch+1} "
        f"Average Loss: {avg_loss:.4f}"
    )

# ------------------------------------------------
# Save fine-tuned model
# ------------------------------------------------

model_save_path = (
    MODELS_DIR
    / "clip_finetuned_fashion"
)

model.save_pretrained(
    model_save_path
)

processor.save_pretrained(
    model_save_path
)

print("\nFine-tuned model saved")

print(model_save_path)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded


Loss: 0.0000: 100%|██████████| 313/313 [39:42<00:00,  7.61s/it]


Epoch 1 Average Loss: 0.0011


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuned model saved
/content/drive/MyDrive/Visual_Product_Search/models/clip_finetuned_fashion


In [16]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm import tqdm
import hnswlib

# ------------------------------------------------
# Device
# ------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

# ------------------------------------------------
# Load fine-tuned CLIP
# ------------------------------------------------

model_path = (
    MODELS_DIR
    / "clip_finetuned_fashion"
)

model = CLIPModel.from_pretrained(
    model_path
).to(device)

processor = CLIPProcessor.from_pretrained(
    model_path
)

print("Fine-tuned CLIP loaded")

# ------------------------------------------------
# Load embeddings
# ------------------------------------------------

embedding_file = (
    EMBEDDINGS_DIR
    / "gallery_embeddings_full.npy"
)

metadata_file = (
    EMBEDDINGS_DIR
    / "gallery_metadata_full.csv"
)

embeddings = np.load(
    embedding_file
)

metadata = pd.read_csv(
    metadata_file
)

print("\nEmbeddings loaded:")
print(embeddings.shape)

# ------------------------------------------------
# Load HNSW index
# ------------------------------------------------

dim = embeddings.shape[1]

index = hnswlib.Index(
    space='cosine',
    dim=dim
)

index.load_index(
    str(
        INDEX_DIR
        / "fashion_hnsw_full.index"
    )
)

index.set_ef(50)

print("\nHNSW index loaded")

# ------------------------------------------------
# Read evaluation file
# ------------------------------------------------

with open(EVAL_FILE, "r") as f:
    lines = f.readlines()[2:]

data = []

for line in lines:

    parts = line.strip().split()

    data.append({
        "image_path": parts[0],
        "item_id": parts[1],
        "split": parts[2]
    })

df = pd.DataFrame(data)

# ------------------------------------------------
# Query subset
# ------------------------------------------------

query_df = (
    df[df["split"] == "query"]
    .head(1000)
)

print("\nQueries selected:")
print(len(query_df))

# ------------------------------------------------
# Recall@5
# ------------------------------------------------

k = 5
correct = 0

for _, row in tqdm(
    query_df.iterrows(),
    total=len(query_df)
):

    try:

        img_path = (
            IMG_ROOT / row["image_path"]
        )

        image = Image.open(
            img_path
        ).convert("RGB")

        # Preprocess
        inputs = processor(
            images=image,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        # Generate embedding
        with torch.no_grad():

            emb = model.get_image_features(
                **inputs
            )

        if hasattr(emb, "pooler_output"):
            emb = emb.pooler_output

        emb = F.normalize(
            emb,
            p=2,
            dim=-1
        )

        emb = emb.cpu().numpy()

        # Search
        labels, distances = index.knn_query(
            emb,
            k=k
        )

        retrieved_items = metadata.iloc[
            labels[0]
        ]["item_id"].values

        # Correct retrieval
        if row["item_id"] in retrieved_items:
            correct += 1

    except Exception as e:

        print(f"Error: {img_path}")

# ------------------------------------------------
# Final Recall
# ------------------------------------------------

recall_at_5 = correct / len(query_df)

print(f"\nFine-Tuned Recall@5: {recall_at_5:.4f}")

Using device: cuda


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Fine-tuned CLIP loaded

Embeddings loaded:
(12612, 512)

HNSW index loaded

Queries selected:
1000


100%|██████████| 1000/1000 [00:16<00:00, 61.14it/s]


Fine-Tuned Recall@5: 0.0020


In [17]:
from pathlib import Path

print("=" * 60)
print("FINAL PROJECT ARTIFACTS")
print("=" * 60)

artifacts = {

    "Full Gallery Embeddings":
        EMBEDDINGS_DIR
        / "gallery_embeddings_full.npy",

    "Full Gallery Metadata":
        EMBEDDINGS_DIR
        / "gallery_metadata_full.csv",

    "Full HNSW Index":
        INDEX_DIR
        / "fashion_hnsw_full.index",

    "Fine-Tuned CLIP Model":
        MODELS_DIR
        / "clip_finetuned_fashion"
}

for name, path in artifacts.items():

    print(f"\n{name}")

    print(path)

    print("Exists:", path.exists())

FINAL PROJECT ARTIFACTS

Full Gallery Embeddings
/content/drive/MyDrive/Visual_Product_Search/embeddings/gallery_embeddings_full.npy
Exists: True

Full Gallery Metadata
/content/drive/MyDrive/Visual_Product_Search/embeddings/gallery_metadata_full.csv
Exists: True

Full HNSW Index
/content/drive/MyDrive/Visual_Product_Search/indexes/fashion_hnsw_full.index
Exists: True

Fine-Tuned CLIP Model
/content/drive/MyDrive/Visual_Product_Search/models/clip_finetuned_fashion
Exists: True
